# ETL — Online Shoppers Purchasing Intention

**Objetivo:** transformar el dataset crudo (ya explorado en `EDA.ipynb`) en un dataset limpio, tipado y documentado, actuando sobre las decisiones que el EDA dejó pendientes (duplicados, outliers) y validando formalmente lo que allí se observó (sin nulos, consistencia lógica entre columnas).

El dataset resultante tiene doble propósito:
1. Insumo canónico para `feature_enginering.ipynb` (encoding, escalado, split train/test — no se hacen acá).
2. Fuente directa para un **dashboard de Power BI** (por eso se agregan columnas con etiquetas legibles y se persiste como CSV).

No se realiza imputación (no hay valores faltantes) ni se generan datos sintéticos en ninguna etapa.

## 1. Configuración y carga de datos

In [1]:
import sys
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)

# Agregamos la raíz del proyecto al path para poder importar `src`
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import cargar_datos
from src.preprocessing import (
    validar_esquema,
    corregir_tipos,
    detectar_duplicados,
    resumen_duplicados,
    eliminar_duplicados,
    detectar_outliers_iqr,
    resumen_outliers,
    validar_consistencia,
    agregar_etiquetas_legibles,
    guardar_datos_procesados,
    RUTA_PROCESSED_POR_DEFECTO,
)

df = cargar_datos()
print(f"Dataset cargado: {df.shape[0]} filas x {df.shape[1]} columnas")
df.head()

Dataset cargado: 12330 filas x 18 columnas


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False,False
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,True,False


Partimos del mismo dataset crudo que `EDA.ipynb`; ver ese notebook para el análisis exploratorio completo (distribución de variables, correlaciones, hallazgos detallados).

## 2. Validación de esquema y valores faltantes

In [2]:
validar_esquema(df)
print("Esquema válido: 18 columnas esperadas presentes y sin valores nulos.")

Esquema válido: 18 columnas esperadas presentes y sin valores nulos.


`validar_esquema` convierte el hallazgo de EDA ("no hay valores faltantes en ninguna columna") en una invariante forzada, no un supuesto: si el CSV fuente cambiara y apareciera algún nulo o alguna columna inesperada, esta celda fallaría de forma explícita en vez de propagar el problema silenciosamente. Como no hay nulos, no se realiza ningún paso de imputación.

## 3. Corrección de tipos de datos

In [3]:
print("Tipos antes de la corrección:")
print(df.dtypes)

Tipos antes de la corrección:
Administrative               int64
Administrative_Duration    float64
Informational                int64
Informational_Duration     float64
ProductRelated               int64
ProductRelated_Duration    float64
BounceRates                float64
ExitRates                  float64
PageValues                 float64
SpecialDay                 float64
Month                          str
OperatingSystems             int64
Browser                      int64
Region                       int64
TrafficType                  int64
VisitorType                    str
Weekend                       bool
Revenue                       bool
dtype: object


In [4]:
df = corregir_tipos(df)
print("Tipos después de la corrección:")
print(df.dtypes)

Tipos después de la corrección:
Administrative                int64
Administrative_Duration     float64
Informational                 int64
Informational_Duration      float64
ProductRelated                int64
ProductRelated_Duration     float64
BounceRates                 float64
ExitRates                   float64
PageValues                  float64
SpecialDay                  float64
Month                      category
OperatingSystems           category
Browser                    category
Region                     category
TrafficType                category
VisitorType                category
Weekend                        bool
Revenue                        bool
dtype: object


`Weekend` y `Revenue` quedan como `bool` (defensivo: ya venían como bool desde el CSV, pero lo hacemos explícito porque los booleanos no sobreviven una vuelta por CSV — importante para cuando se recargue el dataset procesado más adelante). `Month`, `VisitorType`, `OperatingSystems`, `Browser`, `Region` y `TrafficType` pasan a `category`: son códigos/etiquetas, no cantidades continuas, y esto deja claro su rol semántico de cara al encoding que se hará en `feature_enginering.ipynb`.

## 4. Duplicados: decisión y tratamiento

In [5]:
df = detectar_duplicados(df)
resumen_dup = resumen_duplicados(df)
resumen_dup

{'n_filas': 12330, 'n_duplicados_exactos': 201, 'pct_duplicados': 1.63}

**Decisión: se eliminan.** El ETL alimenta tanto el modelo como el dashboard de Power BI, y ambos consumidores deben ver el mismo dataset para que las métricas sean coherentes entre sí. Se marcan primero (`es_duplicado_exacto`, resumen arriba) para dejar auditado cuántos había y qué proporción representan, y luego se eliminan (`eliminar_duplicados`, conservando la primera aparición de cada fila) para que el CSV persistido sea único: sin duplicados para nadie, en vez de dejar la decisión librada a cada consumidor.

> **Nota sobre el conteo:** las 201 filas de arriba usan `keep=False` (marca *todas* las filas de cada grupo duplicado, incluida la primera aparición) — por eso no coincide con las 125 filas reportadas en `EDA.ipynb`, que cuenta con la convención por defecto de pandas (`keep="first"`), es decir, solo las repeticiones *además* de la primera. Ambas cifras describen el mismo fenómeno; las filas efectivamente eliminadas (ver celda siguiente) son 125 en los dos casos.

In [6]:
filas_antes = df.shape[0]
n_duplicados_a_eliminar = df.duplicated(keep="first").sum()
df = eliminar_duplicados(df)
print(f"Filas antes: {filas_antes} -> después de eliminar duplicados: {df.shape[0]}")

Filas antes: 12330 -> después de eliminar duplicados: 12205


## 5. Outliers: decisión y tratamiento

In [7]:
df = detectar_outliers_iqr(df)
tabla_outliers = resumen_outliers(df)
tabla_outliers

,columna,n_outliers,pct_outliers
0,Administrative,404,3.31
1,Administrative_Duration,1149,9.41
2,Informational,2631,21.56
3,Informational_Duration,2405,19.71
4,ProductRelated,1007,8.25
5,ProductRelated_Duration,951,7.79
6,BounceRates,1428,11.70
7,ExitRates,1325,10.86
8,PageValues,2730,22.37
9,SpecialDay,1249,10.23


**Decisión: se marcan (regla IQR, igual que en los boxplots de `EDA.ipynb`), no se capan ni se recortan.** El EDA ya concluyó que las distribuciones sesgadas de `PageValues`, `Informational` e `Informational_Duration` probablemente reflejan comportamiento real de navegación, no errores de captura. Capar/winsorizar es una decisión que depende del modelo que se vaya a entrenar (los modelos basados en árboles son insensibles a esto; los lineales o basados en distancia no) — por eso se deja explícitamente para `feature_enginering.ipynb`. Para un dashboard de negocio también tiene sentido: un analista quiere ver los valores reales, no una versión recortada de la realidad, y puede usar las columnas `outlier_*` para filtrar si lo necesita.

## 6. Chequeos de consistencia y validez

In [8]:
reporte_consistencia = validar_consistencia(df)
reporte_consistencia

,chequeo,n_violaciones,severidad
0,sin_negativos_Administrative,0,dura
1,sin_negativos_Administrative_Duration,0,dura
2,sin_negativos_Informational,0,dura
3,sin_negativos_Informational_Duration,0,dura
4,sin_negativos_ProductRelated,0,dura
5,sin_negativos_ProductRelated_Duration,0,dura
6,sin_negativos_BounceRates,0,dura
7,sin_negativos_ExitRates,0,dura
8,sin_negativos_PageValues,0,dura
9,sin_negativos_SpecialDay,0,dura


Se validan invariantes duras (levantan error si se violan, porque indicarían corrupción real de datos): ningún valor numérico negativo, y ningún caso de conteo de páginas en cero con duración mayor a cero. Ninguna se viola en este dataset.

También se reportan casos informativos (no son errores): sesiones con conteo de páginas positivo pero duración registrada en cero (plausible para visitas muy breves, redondeadas a 0 segundos) — ocurre en 135 filas para `Administrative`, 226 para `Informational` y 592 para `ProductRelated`. Y se confirma que `BounceRates`, `ExitRates` y `SpecialDay` están siempre dentro del rango `[0, 1]` esperado.

## 7. Balance de clases (verificación)

In [9]:
df["Revenue"].value_counts(normalize=True).rename("proporcion")

Revenue
False    0.843671
True     0.156329
Name: proporcion, dtype: float64

Se confirma el desbalance ya observado en el EDA (~85% no compra vs ~15% compra). **No se corrige acá** (ni resampling ni class weights) — es una decisión de modelado que se aplica, si hace falta, únicamente sobre el split de entrenamiento en `modeling_mvp.ipynb`. Tampoco se generan filas sintéticas en ninguna etapa de este ETL: tanto el dataset que alimenta el modelo como el que alimenta el dashboard de Power BI deben reflejar datos reales.

## 8. Etiquetas legibles para Power BI

In [10]:
df = agregar_etiquetas_legibles(df)
df[["Weekend", "Weekend_label", "Revenue", "Revenue_label"]].sample(5, random_state=42)

,Weekend,Weekend_label,Revenue,Revenue_label
5960,False,No,False,No compra
7662,False,No,True,Compra
200,False,No,False,No compra
10351,True,Sí,False,No compra
1153,False,No,False,No compra


Se agregan `Weekend_label` ("Sí"/"No") y `Revenue_label` ("Compra"/"No compra") como columnas adicionales, sin modificar las originales, para que el dashboard sea legible para negocio sin perder los valores que necesita el pipeline de modelado. `OperatingSystems`, `Browser`, `Region` y `TrafficType` quedan como códigos numéricos: el dataset original (UCI) no incluye el diccionario para traducirlos a nombres reales — es una limitación conocida del dataset, no una omisión de este ETL.

## 9. Resultado final

In [11]:
print(f"Shape final: {df.shape[0]} filas x {df.shape[1]} columnas")
assert df.shape[0] == filas_antes - n_duplicados_a_eliminar, (
    "El shape final no coincide con las filas esperadas tras eliminar duplicados"
)
df.columns.tolist()

Shape final: 12205 filas x 30 columnas


['Administrative',
 'Administrative_Duration',
 'Informational',
 'Informational_Duration',
 'ProductRelated',
 'ProductRelated_Duration',
 'BounceRates',
 'ExitRates',
 'PageValues',
 'SpecialDay',
 'Month',
 'OperatingSystems',
 'Browser',
 'Region',
 'TrafficType',
 'VisitorType',
 'Weekend',
 'Revenue',
 'outlier_Administrative',
 'outlier_Administrative_Duration',
 'outlier_Informational',
 'outlier_Informational_Duration',
 'outlier_ProductRelated',
 'outlier_ProductRelated_Duration',
 'outlier_BounceRates',
 'outlier_ExitRates',
 'outlier_PageValues',
 'outlier_SpecialDay',
 'Weekend_label',
 'Revenue_label']

## 10. Persistencia (Load)

In [12]:
ruta_guardada = guardar_datos_procesados(df)
print(f"Dataset procesado guardado en: {ruta_guardada}")
print(f"Tamaño del archivo: {ruta_guardada.stat().st_size / 1024:.1f} KB")

Dataset procesado guardado en: /home/juanma/HENRRY/PROYECTO_FINAL/Proyecto-Final-Henry/data/processed/online_shoppers_intention_procesado.csv
Tamaño del archivo: 1973.2 KB


Este CSV (`data/processed/online_shoppers_intention_procesado.csv`) es el archivo que se conecta como fuente en Power BI, y el que `feature_enginering.ipynb` debería recargar con `cargar_datos_procesados()` (que reaplica automáticamente los tipos `bool`/`category`, ya que un CSV no los preserva).

## 11. Conclusiones del ETL

A partir del dataset crudo validado en `EDA.ipynb`, este ETL:

- **Validó** el esquema y confirmó, como invariante forzada, que no hay valores faltantes (por lo tanto, no se requiere imputación).
- **Corrigió tipos**: `Weekend`/`Revenue` como `bool`, columnas de código como `category`.
- **Decidió y documentó** el tratamiento de duplicados exactos: se marcaron (~1.6% de las filas, `es_duplicado_exacto`) y luego se eliminaron (conservando la primera aparición), para que modelo y dashboard partan del mismo dataset.
- **Decidió y documentó** el tratamiento de outliers (marcados por IQR con columnas `outlier_*`, no capados — se consideran comportamiento real, y el capado depende del modelo que se use más adelante).
- **Verificó consistencia lógica** entre conteos y duraciones, y rangos válidos en las tasas/proximidad a fecha especial — sin violaciones duras.
- **Verificó** (sin corregir) el desbalance de clases de `Revenue`, y **no agregó datos sintéticos** en ninguna etapa.
- **Agregó etiquetas legibles** (`Weekend_label`, `Revenue_label`) pensadas para consumo directo en un dashboard de Power BI.
- **Persistió** el resultado en `data/processed/online_shoppers_intention_procesado.csv`, con 12.205 filas (12.330 originales menos 125 duplicados exactos eliminados).

**Lo que queda para `feature_enginering.ipynb`:** encoding de variables categóricas para modelado, escalado/normalización, capado de outliers (si el modelo elegido lo requiere), features derivadas, y el split train/test (debe hacerse ahí, no acá, para evitar fuga de datos al ajustar encoders/scalers).